# Install Dependecies

In [1]:
%%capture
%pip install -q "transformers>=5,<6"
%pip install matplotlib
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [42]:
import numpy as np
import pandas as pd
import random
import torch
from datasets import load_dataset
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm.std import tqdm
from transformers import AutoTokenizer
from transformers import AutoModel
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier

# Import datasets

In [4]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing

In [5]:
LANGUAGES = ["ar", "ko", "te"]

In [6]:
# select languages
df_train = df_train[df_train["lang"].isin(LANGUAGES)].copy()
df_val = df_val[df_val["lang"].isin(LANGUAGES)].copy()

# Tokeniser

In [7]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 3 Week 37: Structured Span Prediction
* Convert the character-level answer offsets into BIO labels over context tokens. 
* Add automatic checks for at least the following cases: 
  - an answer at character 0, 
  - a multi-token answer, 
  - punctuation adjacent to an answer and an unanswerable example. 
* Document how subword pieces are handled if applicable. 
* Implement one question-conditioned sequence labeller for the group: the representation of the question must influence the predicted label for every context token. 
* Compare it with a simple span baseline. An empty-output baseline is sufficient. If you use lexical overlap, select context tokens using only overlap with the question (optionally after fixed preprocessing or translation), convert the best contiguous run to a span and never use gold answer text or offsets. The correct output for an unanswerable question is an empty span. Evaluate and analyse the models according to Section 1.


### 1. Convert the character-level answer offsets into BIO labels over context tokens

In [8]:
def tokenize_with_offsets(text):
    encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    offsets = [tuple(offset) for offset in encoding["offset_mapping"]]
    return tokens, offsets

In [9]:
# lab_2.ipynb
def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels

In [10]:
# lab_2.ipynb
def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

### Apply to both splits
Unanswerable rows have `answer_start == -1` (their `answer` column is not empty), so we key on `answerable` and give them all-`O` labels.

In [11]:
def row_to_bio(row):
    if not row["answerable"]:
        return character_span_to_bio(row["context"])
    return character_span_to_bio(row["context"], row["answer_start"], row["answer"])

In [12]:
# train
results = [row_to_bio(row) for _, row in df_train.iterrows()]
df_train["tokens"] = [tokens for tokens, offsets, labels in results]
df_train["offsets"] = [offsets for tokens, offsets, labels in results]
df_train["labels"] = [labels for tokens, offsets, labels in results]
df_train

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (547 > 512). Running this sequence through the model will result in indexing errors


,question,context,lang,answerable,answer_start,answer,answer_inlang,tokens,offsets,labels
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None,"[▁The, ▁conflict, ▁between, ▁France, ▁and, ▁Sp...","[(0, 3), (4, 12), (13, 20), (21, 27), (28, 31)...","[O, O, O, B-ANS, O, O, O, O, O, O, O, O, O, O,..."
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None,"[▁X, -, ray, s, ▁make, ▁up, ▁X, -, radi, ation...","[(0, 1), (1, 2), (2, 5), (5, 6), (7, 11), (12,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,None,"[▁In, ▁2022, ,, ▁Beijing, ▁will, ▁become, ▁the...","[(0, 2), (3, 7), (7, 8), (9, 16), (17, 21), (2...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),None,"[▁The, ▁British, ▁Broadcast, ing, ▁Corporation...","[(0, 3), (4, 11), (12, 21), (21, 24), (25, 36)...","[O, B-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-ANS, ..."
4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,None,"[▁Palestin, e, ▁(, ▁', ),, ▁official, ly, ▁the...","[(0, 8), (8, 9), (10, 11), (12, 13), (13, 15),...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
...,...,...,...,...,...,...,...,...,...,...
15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,None,"[▁In, ▁February, ▁2012,, ▁Somali, ▁government,...","[(0, 2), (3, 11), (12, 17), (18, 24), (25, 35)...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,None,"[▁The, ▁first, ▁earth, ▁track, s, ▁were, ▁crea...","[(0, 3), (4, 9), (10, 15), (16, 21), (21, 22),...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,None,"[▁Ab, del, ▁Fat, tah, ▁Sa, eed, ▁Hussein, ▁Kha...","[(0, 2), (2, 5), (6, 9), (9, 12), (13, 15), (1...","[B-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-A..."
15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,None,"[▁Munich, ▁(, ;, ▁;, ▁), ▁is, ▁the, ▁capital, ...","[(0, 6), (7, 8), (8, 9), (10, 11), (12, 13), (...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


In [13]:
# val
results = [row_to_bio(row) for _, row in df_val.iterrows()]
df_val["tokens"] = [tokens for tokens, offsets, labels in results]
df_val["offsets"] = [offsets for tokens, offsets, labels in results]
df_val["labels"] = [labels for tokens, offsets, labels in results]
df_val

,question,context,lang,answerable,answer_start,answer,answer_inlang,tokens,offsets,labels
0,ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?,Portland is the largest city in the U.S. state...,te,True,0,Portland,None,"[▁Portland, ▁is, ▁the, ▁largest, ▁city, ▁in, ▁...","[(0, 8), (9, 11), (12, 15), (16, 23), (24, 28)...","[B-ANS, O, O, O, O, O, O, O, O, O, O, O, O, O,..."
1,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,"The word cholera is from ""kholera"" from χολή ""...",te,True,99,Indian subcontinent,None,"[▁The, ▁word, ▁cho, lera, ▁is, ▁from, ▁"", kho,...","[(0, 3), (4, 8), (9, 12), (12, 16), (17, 19), ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,Since it became widespread in the 19th century...,te,True,451,England,None,"[▁Since, ▁it, ▁became, ▁wide, spre, ad, ▁in, ▁...","[(0, 5), (6, 8), (9, 15), (16, 20), (20, 24), ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,World War I occurred from 1914 to 1918. In ter...,te,True,26,1914,None,"[▁World, ▁War, ▁I, ▁occur, red, ▁from, ▁1914, ...","[(0, 5), (6, 9), (10, 11), (12, 17), (17, 20),...","[O, O, O, O, O, O, B-ANS, O, O, O, O, O, O, O,..."
4,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,"World War I (often abbreviated as WWI or WW1),...",te,True,155,28 July 1914,None,"[▁World, ▁War, ▁I, ▁(, o, ften, ▁ab, brev, i, ...","[(0, 5), (6, 9), (10, 11), (12, 13), (13, 14),...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
...,...,...,...,...,...,...,...,...,...,...
3006,2011 జనగణన ప్రకారం రెయ్యలగడ్ద గ్రామములో పురుషు...,Reyyalagadda is a village belonging to Gangara...,te,True,378,37,37,"[▁Rey, ya, lag, adda, ▁is, ▁a, ▁village, ▁belo...","[(0, 3), (3, 5), (5, 8), (8, 12), (13, 15), (1...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3007,2011 జనాభా లెక్కల ప్రకారం బూతుమిల్లిపాడు గ్రామ...,Boothumillipadu is a village in Gannavaram man...,te,True,308,433,433,"[▁Boot, hu, milli, padu, ▁is, ▁a, ▁village, ▁i...","[(0, 4), (4, 6), (6, 11), (11, 15), (16, 18), ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3008,2011 జనాభా లెక్కల ప్రకారం మల్లవేముల గ్రామ జనాభ...,Mallavemula is a village belonging to Chagalam...,te,False,-1,1131,1131,"[▁Mall, ave, mula, ▁is, ▁a, ▁village, ▁belo, n...","[(0, 4), (4, 7), (7, 11), (12, 14), (15, 16), ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3009,2011 నాటికి రష్యా దేశ ప్రధాన మంత్రి ఎవరు?,"Andria Urushadze (; born April 25, 1968) is a ...",te,False,-1,Vladimir Putin,వ్లాదిమిర్ పుతిన్,"[▁Andri, a, ▁Ur, usha, dze, ▁(, ;, ▁born, ▁Apr...","[(0, 5), (5, 6), (7, 9), (9, 13), (13, 16), (1...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


### Round trip: labels back to a character span

In [14]:
def round_trip(row):
    start, text = bio_to_character_span(row["context"], row["offsets"], row["labels"])
    return text

In [15]:
# train
ans = df_train[df_train["answerable"]]
recovered = [round_trip(row) for _, row in ans.iterrows()]
exact = [text == answer for text, answer in zip(recovered, ans["answer"])]
contains = [answer in text for text, answer in zip(recovered, ans["answer"])]
print(f"checked: {len(ans)}; exact: {sum(exact)}; fail: {contains.count(False)}")

checked: 5972; exact: 5598; fail: 0


In [16]:
# val
ans = df_val[df_val["answerable"]]
recovered = [round_trip(row) for _, row in ans.iterrows()]
exact = [text == answer for text, answer in zip(recovered, ans["answer"])]
contains = [answer in text for text, answer in zip(recovered, ans["answer"])]
print(f"checked: {len(ans)}; exact: {sum(exact)}; fail: {contains.count(False)}")

checked: 991; exact: 941; fail: 0


### Label distribution

In [17]:
# from lab_2.ipynb: "Audit the label distribution" (adapted to string labels)
# train
label_counts = pd.Series([label for labels in df_train["labels"] for label in labels]).value_counts()
display(pd.DataFrame({"count": label_counts, "proportion": label_counts / label_counts.sum()}))

,count,proportion
O,890224,0.966669
I-ANS,24723,0.026846
B-ANS,5972,0.006485


In [18]:
# val
label_counts = pd.Series([label for labels in df_val["labels"] for label in labels]).value_counts()
display(pd.DataFrame({"count": label_counts, "proportion": label_counts / label_counts.sum()}))

,count,proportion
O,170678,0.972203
I-ANS,3889,0.022152
B-ANS,991,0.005645


In [19]:
row = df_train[df_train["answerable"]].iloc[0]
start = row["labels"].index("B-ANS")
print(row["question"] + " | " + row["answer"])
display(pd.DataFrame({"token": row["tokens"], "offset": row["offsets"], "label": row["labels"]}).iloc[start - 3:start + 4])

30년 전쟁의 승자는 누구인가? | France


,token,offset,label
0,▁The,"(0, 3)",O
1,▁conflict,"(4, 12)",O
2,▁between,"(13, 20)",O
3,▁France,"(21, 27)",B-ANS
4,▁and,"(28, 31)",O
5,▁Spain,"(32, 37)",O
6,▁continued,"(38, 47)",O


### 2. Automatic checks
1. An answer at character 0
2. A multi-token answer
3. Punctuation adjacent to an answer
4. An unanswerable example.

In [63]:
# check 1: answer at character 0
context = "NLP is the best course at DIKU."
answer = "NLP"
tokens, offsets, labels = character_span_to_bio(context, 0, answer)
print(116 * "-")
print("check 1:  answer at character 0")
print("context: ", context)
print("answer:  ", answer)
print("tokens:  ", tokens)
print("offsets: ", offsets)
print("labels:  ", labels)
print(116 * "-")
assert tokens == ["▁N", "LP", "▁is", "▁the", "▁best", "▁course", "▁at", "▁D", "IKU", "."]
assert labels == ["B-ANS", "I-ANS", "O", "O", "O", "O", "O", "O", "O", "O"]
assert bio_to_character_span(context, offsets, labels) == (0, answer)

# check 2: multi-token answer
context = "Rumour has it that Natural language processing is the only course where the homework talks back."
answer = "Natural language processing"
tokens, offsets, labels = character_span_to_bio(context, context.index(answer), answer)
print("check 2:  multi-token answer")
print("context: ", context)
print("answer:  ", answer)
print("tokens:  ", tokens)
print("offsets: ", offsets)
print("labels:  ", labels)
print(116 * "-")
assert tokens[5:9] == ["▁Natural", "▁language", "▁process", "ing"]
assert labels[5:9] == ["B-ANS", "I-ANS", "I-ANS", "I-ANS"]
assert labels[:5] == ["O"] * 5 and labels[9:] == ["O"] * 12
assert bio_to_character_span(context, offsets, labels) == (context.index(answer), answer)

# check 3: punctuation adjacent to the answer
context = "The tokenizer was last seen in Copenhagen, crying over Telegu tokens."
answer = "Copenhagen"
tokens, offsets, labels = character_span_to_bio(context, context.index(answer), answer)
print("check 3:  punctuation adjacent to the answer")
print("context: ", context)
print("answer:  ", answer)
print("tokens:  ", tokens)
print("offsets: ", offsets)
print("labels:  ", labels)
print(116 * "-")
assert tokens[8:10] == ["▁Copenhagen", ","]
assert labels[8:10] == ["B-ANS", "O"]
assert labels.count("B-ANS") == 1 and labels.count("I-ANS") == 0
assert bio_to_character_span(context, offsets, labels) == (context.index(answer), answer)

# check 4: unanswerable example
tokens, offsets, labels = character_span_to_bio(context)
print("check 4:  unanswerable example")
print("context: ", context)
print("answer:  ", answer)
print("tokens:  ", tokens)
print("offsets: ", offsets)
print("labels:  ", labels)
assert labels == ["O"] * len(tokens)
assert bio_to_character_span(context, offsets, labels) == (None, "")
print(116 * "-")

--------------------------------------------------------------------------------------------------------------------
check 1:  answer at character 0
context:  NLP is the best course at DIKU.
answer:   NLP
tokens:   ['▁N', 'LP', '▁is', '▁the', '▁best', '▁course', '▁at', '▁D', 'IKU', '.']
offsets:  [(0, 1), (1, 3), (4, 6), (7, 10), (11, 15), (16, 22), (23, 25), (26, 27), (27, 30), (30, 31)]
labels:   ['B-ANS', 'I-ANS', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
--------------------------------------------------------------------------------------------------------------------
check 2:  multi-token answer
context:  Rumour has it that Natural language processing is the only course where the homework talks back.
answer:   Natural language processing
tokens:   ['▁Rum', 'our', '▁has', '▁it', '▁that', '▁Natural', '▁language', '▁process', 'ing', '▁is', '▁the', '▁only', '▁course', '▁where', '▁the', '▁home', 'work', '▁talk', 's', '▁back', '.']
offsets:  [(0, 3), (3, 6), (7, 10), (11, 13), (14, 18), 

### 3. Implement one question-conditioned sequence labeller for the group
The representation of the question must influence the predicted label for every context token. 

In [43]:
# lab_2.ipynb
BIO_TAGS = ["O", "B-ANS", "I-ANS"]
TAG_TO_ID = {tag: index for index, tag in enumerate(BIO_TAGS)}

In [ ]:
# turns the question into a set of its XLM-R pieces so that each context token can be checked if it also occurs in the question
def question_pieces_of(question):
    pieces = tokenizer.tokenize(question)
    res = set(pieces) | {piece.lower() for piece in pieces}
    print(res)
    return res

In [ ]:
# ar question word first in a sentence (reading direction: right-to-left)
# summarises the question type as its first and last piece so that the question word can be paired with each context token shape
def question_words_of(question):
    pieces = [piece for piece in tokenizer.tokenize(question) if piece not in {"?", "؟"}]
    return pieces[0] + " | " + pieces[-1]

In [ ]:
# lab_2.ipynb
def token_features(tokens, index, question_pieces, question_words):
    word = tokens[index]
    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word.prefix2": word[:2].lower(),
        "word.suffix2": word[-2:].lower(),
        "word.suffix3": word[-3:].lower(),
        "word.istitle": word.istitle(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "contains_hyphen": "-" in word,
        "word.in_question": word in question_pieces,  # added feature
        "word.lower_in_question": word.lower() in question_pieces,  # added feature
        "question_words|word.isdigit": f"{question_words}|{word.isdigit()}",  # added feature
        "question_words|word.istitle": f"{question_words}|{word.istitle()}",  # added feature
    }

    if index == 0:
        features["BOS"] = True
    else:
        previous = tokens[index - 1]
        features.update({
            "previous.lower": previous.lower(),
            "previous.istitle": previous.istitle(),
            "previous.isupper": previous.isupper(),
            "previous.in_question": previous in question_pieces,
        })

    if index == len(tokens) - 1:
        features["EOS"] = True
    else:
        following = tokens[index + 1]
        features.update({
            "next.lower": following.lower(),
            "next.istitle": following.istitle(),
            "next.isupper": following.isupper(),
            "next.in_question": following in question_pieces,
        })

    return features

In [ ]:
# lab_2.ipynb
def featurize(df):
    features, labels, lengths = [], [], []
    for tokens, row_labels, question in zip(df["tokens"], df["labels"], df["question"]):
        question_pieces = question_pieces_of(question)
        question_words = question_words_of(question)
        lengths.append(len(tokens))
        features.extend(token_features(tokens, index, question_pieces, question_words) for index in range(len(tokens)))
        labels.extend(TAG_TO_ID[label] for label in row_labels)
    return features, np.asarray(labels), lengths

In [68]:
# lab_2.ipynb
def split_by_lengths(values, lengths):
    sequences = []
    offset = 0
    for length in lengths:
        sequences.append(list(values[offset:offset + length]))
        offset += length
    assert offset == len(values)
    return sequences

In [69]:
# train
train_features, train_labels, train_lengths = featurize(df_train)

{'년', '인가', '의', '▁승', '▁전쟁', '자는', '▁30', '?', '▁누구'}
{'▁발견', '은', '하였', '선', '?', '엑스', '는', '▁', '▁누가', '가'}
{'이', '의', '올림픽', '?', '▁아', '나요', '▁가장', '에서', '네', '▁', '테', '▁최근', '▁올', '▁언제', '렸'}
{'▁가장', '에서', '된', '▁방송', '▁오래', '사는', '▁무엇인가', '?', '▁세상'}
{'▁수도', '인', '스타', '?', '는', '▁팔', '가요', '레', '▁어', '딘'}
{'▁많은', '▁중', '자리', '▁이루어', '리는', '?', '▁가장', '▁', '자', '▁무엇인가', '▁별', '별로', '진'}
{'▁에너지', '▁큰', '?', '▁세상', '▁가장', '는', '에서', '소', '▁발전', '▁무엇인가', '▁풍', '력'}
{'이', '은', '의', '▁15', '?', '▁루', '▁본', '명', '세', '▁무엇인가'}
{'전', '이', '텔', '된', '은', '인가', '?', '▁기업', '비', '▁처음', '▁출시', '▁', '레', '▁컬러', '▁어디'}
{'국', '▁초대', '아', '인가', '의', '제', '우리', '?', '▁누구', '는', '▁황', '▁제', '▁마'}
{'틀', '▁무슨', '▁관계', '였', '?', '는', '▁', '▁힘', '와', '히', '가', '러'}
{'율', '▁어떤', '▁실', '▁구성', '들', '▁있는', '콘', '가', '주기', '은', '에', '되어', '표', '?', '로', '▁', '▁원', '소', '리'}
{'▁일', '제', '?', '는', '점', '▁끝', '▁언제', '기는', '가', '강', '났'}
{'차', '이', '▁많은', '즈', '?', '랜', '▁외', '▁세상', '▁가장', '는', '에서', '▁프', 

In [70]:
# val
validation_features, validation_labels, validation_lengths = featurize(df_val)

{'రె', 'న్', '▁?', 'ం', 'గా', '▁రాష్ట్రంలో', 'పె', '▁నగర', '▁అతి', 'ద్ద', '▁ఏది', '▁ఒ', 'ని'}
{'రా', '▁క', 'ను', '▁మొదట', '▁దేశంలో', 'గ', '▁?', 'న్నారు', 'ొ', 'గా', '▁వ్యాధి', '▁కల', '▁ఏ', 'ని'}
{'రా', '▁క', 'ను', '▁మొదట', '▁దేశంలో', 'గ', '▁?', 'న్నారు', 'ొ', 'గా', '▁వ్యాధి', '▁కల', '▁ఏ', 'ని'}
{'యింది', '▁మొద', 'ల', '▁ఎప్పుడు', '▁ప్రపంచ', '▁యుద్ధం', '▁?', '▁మొదటి'}
{'యింది', '▁మొద', 'ల', '▁ఎప్పుడు', '▁ప్రపంచ', '▁యుద్ధం', '▁?', '▁మొదటి'}
{'▁చట్టం', '▁ఆ', '?', 'మో', '▁సమాచార', 'ించింది', '▁ను', '▁ఏ', 'ద', '▁ప్రభుత్వం', '-2005'}
{'లో', 'దేశం', 'చీ', 'న', 'క', '▁ఉండే', '?', 'వి', '▁ఎన్ని', '▁ప్రా', '▁', 'వాడు', '▁భారత', '▁భాష', 'లు'}
{'గ', '▁స', '▁మనిషి', 'న', '?', 'ం', 'టు', '▁ఉండ', 'ాల్సిన', '▁ఎంత', '▁రక్త', 'ని'}
{'▁?', '▁మానవ', 'తం', 'స', '▁రాష్ట్రంలో', 'టెక్', '▁', 'మి', 'పె', '▁నిర్', '▁అతి', 'ద్ద', '▁ఏది', 'ని', 'స్'}
{'▁క', 'స్సు', 'లో', '▁హక్కు', 'డానికి', '▁పొంద', 'దేశం', '▁వ', '?', 'స', 'నీ', 'టు', '▁భారత', '▁ఉండ', '▁ఎంత', 'వలసిన', '▁ఓ', 'య'}
{'▁క', 'ను', 'లో', 'గ', 'న్నారు', '

In [54]:
# lab_2.ipynb
vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(train_features)
X_validation = vectorizer.transform(validation_features)

for X in (X_train, X_validation):
    X.indices = X.indices.astype(np.int32)
    X.indptr = X.indptr.astype(np.int32)

token_classifier = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    class_weight="balanced",
    max_iter=30,
    random_state=42,
)
token_classifier.fit(X_train, train_labels)

,"<a class=""param-doc-link"" style=""anchor-name: --doc-link-loss;"" rel=""noreferrer"" target=""_blank"" href=""https://scikit-learn.org/1.9/modules/generated/sklearn.linear_model.SGDClassifier.html#:~:text=loss,-%7B%27hinge%27%2C%20%27log_loss%27%2C%20%27modified_huber%27%2C%20%27squared_hinge%27%2C%20%20%20%20%20%20%20%20%27perceptron%27%2C%20%27squared_error%27%2C%20%27huber%27%2C%20%27epsilon_insensitive%27%2C%20%20%20%20%20%20%20%20%27squared_epsilon_insensitive%27%7D%2C%20default%3D%27hinge%27""> loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide<sgd_mathematical_formulation>` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'log_loss'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",1e-05
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",30
,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.Integer values must be in the range `[0, 2**32 - 1]`.",42
,"class_weight class_weight: dict, {class_label: weight} or ""balanced"", default=NonePreset for the class_weight fit parameter.Weights associated with classes. If not given, all classesare supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depe

In [ ]:
# lab_2.ipynb
def valid_bio_transition(previous_id, current_id):
    current = BIO_TAGS[int(current_id)]
    if not current.startswith("I-"):
        return True
    if previous_id is None:
        return False
    entity_type = current[2:]
    return BIO_TAGS[int(previous_id)] in {f"B-{entity_type}", f"I-{entity_type}"}

In [56]:
# lab_2.ipynb
def constrained_beam_decode(token_log_probs, class_ids, beam_size=4):
    beam = [([], 0.0)]

    for scores in token_log_probs:
        candidates = []
        for sequence, sequence_score in beam:
            previous = sequence[-1] if sequence else None
            for column, label_id in enumerate(class_ids):
                if valid_bio_transition(previous, label_id):
                    candidates.append((
                        sequence + [int(label_id)],
                        sequence_score + float(scores[column]),
                    ))
        candidates.sort(key=lambda item: item[1], reverse=True)
        beam = candidates[:beam_size]

    return beam[0][0]

In [ ]:
# keeps only the first predicted span and sets all other labels to O
def first_span(labels):
    kept = ["O"] * len(labels)
    if "B-ANS" not in labels:
        return kept
    index = labels.index("B-ANS")
    kept[index] = "B-ANS"
    index += 1
    while index < len(labels) and labels[index] == "I-ANS":
        kept[index] = "I-ANS"
        index += 1
    return kept

In [ ]:
# val
# predict
validation_log_probs = token_classifier.predict_log_proba(X_validation)
log_prob_sequences = split_by_lengths(validation_log_probs, validation_lengths)

beam_predictions = [
    constrained_beam_decode(scores, token_classifier.classes_, beam_size=4)
    for scores in log_prob_sequences
]

df_val["pred_labels"] = [first_span([BIO_TAGS[label] for label in sequence]) for sequence in beam_predictions]
df_val["pred_answer"] = [bio_to_character_span(context, offsets, labels)[1] for context, offsets, labels in zip(df_val["context"], df_val["offsets"], df_val["pred_labels"])]

print("predicted empty:", round((df_val["pred_answer"] == "").mean(), 3))
display(df_val[df_val["answerable"]][["lang", "question", "answer", "pred_answer"]].head(15))

predicted empty: 0.137


,lang,question,answer,pred_answer
0,te,ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?,Portland,
1,te,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,Indian subcontinent,1817
2,te,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,England,1847
3,te,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,1914,1914
4,te,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,28 July 1914,28
5,te,సమాచార చట్టం-2005 ను ఏ ప్రభుత్వం ఆమోదించింది?,India,11
6,te,ప్రాచీన భారతదేశంలో ఎన్ని భాషలు వాడుకలో ఉండేవి?,122,122
7,te,మనిషిని సగటున ఉండాల్సిన రక్తం ఎంత?,approximately 5 liters,A
8,te,టెక్సస్ రాష్ట్రంలోని అతిపెద్ద మానవ నిర్మితం ఏది ?,JPMorgan Chase Tower,1960
9,te,భారతదేశంలో ఓటు హక్కు పొందడానికి ఉండవలసిన కనీస ...,18,Six
